# WM-811K 설명 방법 비교 · Gradient SHAP 검증

배포된 ce_sqrt_balanced 체크포인트를 직접 업로드하고 SHA-256을 확인한 뒤, 고정 test 클래스 균형 표본(클래스별 최대 50개, seed 42)에서 Grad-CAM, 0 기준 Integrated Gradients, Gradient SHAP을 비교합니다.

- Gradient SHAP은 대시보드와 같은 방식입니다: 같은 웨이퍼 형상에서 모든 활성 die가 정상인 기준, 층화 Expected Gradients, 사후 보정하지 않은 원시 attribution, 로짓 변화의 5% 가산성 점검(미통과 시 표본 64→128→256).
- 기존 die 제거(→웨이퍼 밖) 점검과 함께, Gradient SHAP 기준과 같은 방향인 결함 die→정상 die 교란 점검을 세 방법 모두에 적용합니다.
- 과거 `xai_deployed_model_results.zip`은 IG·Grad-CAM만 비교한 결과이며 Gradient SHAP 검증이 아닙니다. 이 노트북의 결과는 `xai_gradient_shap_validation_results.zip`으로 따로 저장합니다.

모델 선택이나 임계값 조정에는 사용하지 않습니다. 로컬에 `wafer_maps_64.npy`가 있으면 같은 스크립트를 `--device cpu`로 실행해도 됩니다.

In [ ]:
!pip install -q pandas==2.2.3 matplotlib==3.10.0
from google.colab import drive, files
from pathlib import Path
import json, shutil, subprocess, sys, torch
drive.mount('/content/drive')
assert torch.cuda.is_available(), '런타임을 T4 GPU로 변경한 뒤 다시 실행하세요.'
print('GPU:', torch.cuda.get_device_name(0))

아래 셀에서 평가 스크립트, split_assignments.csv, 결과물/wm811k/selected_model_results/best_model.pt를 한 개씩 차례대로 선택합니다.

In [ ]:
print('1/3: evaluate_wm811k_xai_methods.py를 선택하세요.')
script_upload = files.upload()
script_name, script_bytes = next(iter(script_upload.items()))
assert script_name.endswith('.py'), 'Python 평가 스크립트를 선택하세요.'
SCRIPT = Path('/content/evaluate_wm811k_xai_methods.py')
SCRIPT.write_bytes(script_bytes)
print('2/3: split_assignments.csv를 선택하세요.')
split_upload = files.upload()
split_name, split_bytes = next(iter(split_upload.items()))
assert split_name.endswith('.csv'), 'split_assignments.csv를 선택하세요.'
ASSIGNMENTS = Path('/content/split_assignments.csv')
ASSIGNMENTS.write_bytes(split_bytes)
print('3/3: selected_model_results/best_model.pt를 선택하세요.')
checkpoint_upload = files.upload()
assert len(checkpoint_upload) == 1, '체크포인트 한 개만 선택하세요.'
checkpoint_name, checkpoint_bytes = next(iter(checkpoint_upload.items()))
assert checkpoint_name.endswith('.pt'), '배포 best_model.pt를 선택하세요.'
CHECKPOINT = Path('/content/deployed_best_model.pt')
CHECKPOINT.write_bytes(checkpoint_bytes)
import hashlib
EXPECTED_CHECKPOINT_SHA256 = '77ea8394f593ccd0e1ffa85ce1b8e59c52fcd44b109d7e3587c66f7290d4557b'
assert hashlib.sha256(checkpoint_bytes).hexdigest() == EXPECTED_CHECKPOINT_SHA256, '현재 배포 체크포인트와 다른 파일입니다.'
ASSIGNMENTS_RAW_SHA256 = hashlib.sha256(split_bytes).hexdigest()
ASSIGNMENTS_CANONICAL_SHA256 = hashlib.sha256(split_bytes.replace(b'\r\n', b'\n').replace(b'\r', b'\n')).hexdigest()
print('분할표 원시 SHA-256:', ASSIGNMENTS_RAW_SHA256)
print('분할표 줄바꿈 정규화 SHA-256:', ASSIGNMENTS_CANONICAL_SHA256)
print('업로드 완료')

In [ ]:
DATA_DIR = Path('/content/drive/MyDrive/AI_project/WM811K_processed')
MAPS = DATA_DIR / 'wafer_maps_64.npy'
LABELS = DATA_DIR / 'labels.npy'
for path in (MAPS, LABELS, CHECKPOINT, ASSIGNMENTS, SCRIPT):
    assert path.is_file(), f'필수 파일이 없습니다: {path}'
OUTPUT_DIR = Path('/content/drive/MyDrive/AI_project/WM811K_xai_comparison/deployed_gradient_shap')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
command = [sys.executable, str(SCRIPT), '--maps', str(MAPS), '--labels', str(LABELS), '--assignments', str(ASSIGNMENTS), '--checkpoint', str(CHECKPOINT), '--expected-checkpoint-sha256', EXPECTED_CHECKPOINT_SHA256, '--output-dir', str(OUTPUT_DIR), '--device', 'cuda', '--temperature', '1.047171711723721', '--per-class', '50', '--ig-steps', '24', '--mask-fraction', '0.10', '--random-repeats', '5', '--seed', '42', '--gradient-shap-samples', '64', '--gradient-shap-max-samples', '256']
subprocess.run(command, check=True)
summary = json.loads((OUTPUT_DIR / 'xai_method_summary.json').read_text(encoding='utf-8'))
assert summary['schema_version'] == 3 and 'Gradient SHAP' in summary['methods']
assert summary['assignments_canonical_sha256'] == ASSIGNMENTS_CANONICAL_SHA256
print('Gradient SHAP 원시 가산성 통과율:', summary['gradient_shap']['additivity_pass_rate'])
print('XAI 비교 완료:', OUTPUT_DIR)

In [ ]:
ZIP_BASE = Path('/content/xai_gradient_shap_validation_results')
zip_path = Path(shutil.make_archive(str(ZIP_BASE), 'zip', root_dir=OUTPUT_DIR))
print('다운로드 파일:', zip_path, f'{zip_path.stat().st_size / 1024**2:.2f} MB')
files.download(str(zip_path))

## 결과 가져오기

내려받은 ZIP의 파일을 로컬 프로젝트의 `결과물/wm811k/xai_gradient_shap_validation/`에 풀고 다음으로 검증합니다. 검증을 통과해야 대시보드 `성능·검증 → SHAP 검증`에 현재 배포 설명법의 증거로 표시됩니다.

```powershell
.\.venv\Scripts\python.exe .\코드\wm811k\validate_wm811k_gradient_shap_results.py --result-dir .\결과물\wm811k\xai_gradient_shap_validation --assignments .\결과물\wm811k\split_results\split_assignments.csv --checkpoint .\결과물\wm811k\selected_model_results\best_model.pt
```